# Reintegration Readiness Classifier
**Lighthouse Sanctuary — ML Pipeline**

Predict whether a resident is ready for reintegration into family/community based on program progress indicators.

---
## 1. Problem Framing

### Business Problem
Social workers must decide when a resident is ready to transition back to family or community. This decision is currently based on qualitative case assessment, which is time-intensive and potentially inconsistent. A readiness score gives social workers an objective second opinion and helps prioritize case review resources.

### ML Task
- **Type:** Binary classification  
- **Target:** `reintegration_ready` — 1 if reintegration_status == 'Completed' OR current_risk_level improved from initial  
- **Unit of analysis:** One row per resident  
- **Success metric:** ROC-AUC ≥ 0.70; high recall on the positive class (don't miss ready residents)

### Features
| Feature | Source |
|---|---|
| `avg_health_score_trend` | health_wellbeing_records |
| `avg_education_progress` | education_records |
| `incident_frequency` | incident_reports |
| `progress_noted_rate` | process_recordings |
| `counseling_session_count` | process_recordings |
| `days_in_program` | residents |
| `initial_risk_level` | residents |
| `sub_cat_trafficked`, `sub_cat_physical_abuse`, `sub_cat_sexual_abuse` | residents |

### Models compared
1. Logistic Regression (interpretable, good for clinical-style decisions)  
2. Random Forest (captures non-linear interactions)  

### Deployment target
- **Output:** `models/reintegration_model.pkl`  
- **API:** `POST /predict/reintegration` → `{readiness_score, recommendation}`

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NB_DIR    = Path().resolve()
ML_DIR    = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR / 'ml-pipelines'
DATA_DIR  = ML_DIR / 'data'
MODEL_DIR = ML_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

RESIDENT_MASTER = DATA_DIR / 'resident_master.csv'

if not RESIDENT_MASTER.exists():
    print('Building master datasets...')
    subprocess.run([sys.executable, str(ML_DIR / 'master_dataset_builder.py')], check=True)

df = pd.read_csv(RESIDENT_MASTER, low_memory=False)
print(f'resident_master shape: {df.shape}')
df.head()

In [ ]:
# Target distribution
print('Target distribution (reintegration_ready):')
print(df['reintegration_ready'].value_counts())
print(f'Positive rate: {df["reintegration_ready"].mean():.1%}')

print('\nReintegration status breakdown:')
print(df['reintegration_status'].value_counts())

print('\nRisk level transitions:')
print(df.groupby(['initial_risk_level','current_risk_level']).size().unstack(fill_value=0))

In [ ]:
# Target distribution plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Target
counts = df['reintegration_ready'].value_counts()
axes[0].bar(['Not Ready (0)', 'Ready (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Reintegration Ready Distribution')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.5, f'{v} ({v/len(df):.1%})', ha='center')

# Health score trend by readiness
df.boxplot(column='avg_health_score_trend', by='reintegration_ready',
           ax=axes[1], patch_artist=True)
axes[1].set_title('Health Score Trend by Readiness')
axes[1].set_xlabel('Reintegration Ready')
plt.sca(axes[1])
plt.title('Health Score Trend by Readiness')

# Days in program by readiness
df.boxplot(column='days_in_program', by='reintegration_ready',
           ax=axes[2], patch_artist=True)
plt.sca(axes[2])
plt.title('Days in Program by Readiness')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Case category subtype analysis
sub_cat_cols = [c for c in df.columns if c.startswith('sub_cat_')]
print('Available sub-category columns:', sub_cat_cols)

if sub_cat_cols:
    # Convert to numeric (True/False strings)
    sub_df = df[sub_cat_cols].copy()
    for c in sub_cat_cols:
        sub_df[c] = sub_df[c].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0)
    sub_df['reintegration_ready'] = df['reintegration_ready']
    
    # Churn rate for each sub-category
    sub_rates = {}
    for c in sub_cat_cols:
        group = sub_df[sub_df[c] == 1]['reintegration_ready']
        sub_rates[c.replace('sub_cat_','')] = group.mean() if len(group) > 0 else 0
    
    sub_series = pd.Series(sub_rates).sort_values()
    plt.figure(figsize=(10, 4))
    sub_series.plot(kind='barh', color='#3498db')
    plt.axvline(df['reintegration_ready'].mean(), color='red', linestyle='--', label='Overall avg')
    plt.title('Reintegration Ready Rate by Case Sub-Category')
    plt.xlabel('Ready Rate')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap
feature_cols_numeric = [
    'reintegration_ready', 'avg_health_score_trend', 'avg_education_progress',
    'incident_frequency', 'progress_noted_rate', 'counseling_session_count',
    'days_in_program', 'avg_health_score', 'high_severity_count'
]
existing = [c for c in feature_cols_numeric if c in df.columns]
corr = df[existing].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation — Resident Master')
plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, auc, ConfusionMatrixDisplay
import joblib

# Prepare features
NUMERIC_FEATURES = [
    'avg_health_score_trend', 'avg_education_progress', 'incident_frequency',
    'progress_noted_rate', 'counseling_session_count', 'days_in_program'
]
CATEGORICAL_FEATURES = ['initial_risk_level']
BOOL_FEATURES = ['sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse']
TARGET = 'reintegration_ready'

# Encode booleans
model_df = df.copy()
for c in BOOL_FEATURES:
    if c in model_df.columns:
        model_df[c] = model_df[c].astype(str).str.lower().map({'true':1,'false':0,'1':1,'0':0}).fillna(0)

all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [c for c in BOOL_FEATURES if c in model_df.columns]
model_df = model_df[all_features + [TARGET]].dropna(subset=[TARGET])

print(f'Model dataset: {model_df.shape}')
print(f'Positive rate: {model_df[TARGET].mean():.1%}')

In [ ]:
# Build preprocessor
bool_cols_available = [c for c in BOOL_FEATURES if c in model_df.columns]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Medium')),
    ('encoder', OrdinalEncoder(
        categories=[['Low', 'Medium', 'High', 'Critical']],
        handle_unknown='use_encoded_value', unknown_value=1
    ))
])

transformers = [
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
]
if bool_cols_available:
    transformers.append((
        'bool',
        SimpleImputer(strategy='constant', fill_value=0),
        bool_cols_available
    ))

preprocessor = ColumnTransformer(transformers)

X = model_df[all_features]
y = model_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# Define models
models = {
    'Logistic Regression': Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(
            n_estimators=200, max_depth=6, class_weight='balanced',
            random_state=42, n_jobs=-1
        ))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print('5-fold CV ROC-AUC:')
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:30s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# Fit models
fitted_models = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline
    print(f'Fitted: {name}')

---
## 4. Evaluation & Interpretation

In [ ]:
# ROC curves
plt.figure(figsize=(8, 6))
test_aucs = {}
colors = ['#3498db', '#2ecc71']

for (name, pipeline), color in zip(fitted_models.items(), colors):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    test_aucs[name] = roc_auc
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')

plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Reintegration Readiness')
plt.legend()
plt.tight_layout()
plt.show()

best_model_name = max(test_aucs, key=test_aucs.get)
print(f'Best model: {best_model_name} (AUC={test_aucs[best_model_name]:.4f})')

In [ ]:
# Detailed evaluation — best model
best_pipeline = fitted_models[best_model_name]
y_pred = best_pipeline.predict(X_test)

print(f'Classification Report — {best_model_name}')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Not Ready', 'Ready']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Not Ready', 'Ready'],
    cmap='Greens', ax=ax
)
ax.set_title(f'Confusion Matrix — {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP feature importance (Random Forest)
try:
    import shap
    rf_pipeline = fitted_models['Random Forest']
    X_test_transformed = rf_pipeline.named_steps['prep'].transform(X_test)
    rf_clf = rf_pipeline.named_steps['clf']
    
    explainer = shap.TreeExplainer(rf_clf)
    shap_values = explainer.shap_values(X_test_transformed)
    
    # Get feature names after transformation
    feature_names = all_features  # approximate names
    
    plt.figure(figsize=(10, 6))
    if isinstance(shap_values, list):
        sv = shap_values[1]  # positive class
    else:
        sv = shap_values
    
    shap.summary_plot(
        sv, X_test_transformed,
        feature_names=feature_names[:X_test_transformed.shape[1]],
        show=False
    )
    plt.title('SHAP Feature Importance — Reintegration Readiness')
    plt.tight_layout()
    plt.show()
    print('SHAP plot generated.')
except Exception as e:
    print(f'SHAP skipped: {e}')
    # Fallback: RF feature importances
    rf_clf = fitted_models['Random Forest'].named_steps['clf']
    feat_imp = pd.Series(rf_clf.feature_importances_,
                         index=all_features[:len(rf_clf.feature_importances_)]).sort_values()
    feat_imp.plot(kind='barh', figsize=(8, 5), color='#2ecc71')
    plt.title('Random Forest Feature Importances')
    plt.tight_layout()
    plt.show()

In [ ]:
# Logistic Regression coefficients
lr_clf = fitted_models['Logistic Regression'].named_steps['clf']
coef_df = pd.DataFrame({
    'Feature': all_features[:len(lr_clf.coef_[0])],
    'Coefficient': lr_clf.coef_[0]
}).sort_values('Coefficient')

plt.figure(figsize=(8, 5))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('LR Coefficients — Positive = Increases Readiness')
plt.tight_layout()
plt.show()

---
## 5. Causal and Relationship Analysis

In [ ]:
# How does readiness vary by initial risk level?
risk_analysis = df.groupby('initial_risk_level').agg(
    ready_rate=('reintegration_ready', 'mean'),
    count=('resident_id', 'count'),
    avg_days=('days_in_program', 'mean'),
    avg_incidents=('incident_count', 'mean')
).loc[['Low', 'Medium', 'High', 'Critical'], :]

print('=== Readiness by Initial Risk Level ===')
print(risk_analysis.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(risk_analysis.index, risk_analysis['ready_rate'],
       color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'])
ax.set_title('Readiness Rate by Initial Risk Level')
ax.set_ylabel('Readiness Rate')
ax.set_xlabel('Initial Risk Level')
plt.tight_layout()
plt.show()

In [ ]:
# Relationship: education progress vs. health score trend — colored by readiness
plt.figure(figsize=(8, 5))
for label, color, marker in [(0, '#e74c3c', 'x'), (1, '#2ecc71', 'o')]:
    subset = df[df['reintegration_ready'] == label]
    plt.scatter(
        subset['avg_education_progress'].fillna(0),
        subset['avg_health_score_trend'].fillna(0),
        c=color, marker=marker, alpha=0.5, s=30,
        label='Ready' if label == 1 else 'Not Ready'
    )
plt.xlabel('Avg Education Progress (%)')
plt.ylabel('Health Score Trend (latest - first)')
plt.title('Education Progress vs. Health Trend by Readiness')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Effect of counseling sessions on readiness (controlling for days in program)
session_bins = pd.cut(df['counseling_session_count'].fillna(0),
                       bins=[0, 5, 15, 30, 100], labels=['1-5', '6-15', '16-30', '30+'])
session_analysis = df.groupby(session_bins, observed=True)['reintegration_ready'].mean()
print('Readiness by Counseling Session Count:')
print(session_analysis.to_string())

plt.figure(figsize=(7, 4))
session_analysis.plot(kind='bar', color='#3498db')
plt.title('Readiness Rate by Counseling Session Count')
plt.ylabel('Readiness Rate')
plt.xlabel('Session Count Bracket')
plt.xticks(rotation=0)
plt.axhline(df['reintegration_ready'].mean(), color='red', linestyle='--', label='Overall avg')
plt.legend()
plt.tight_layout()
plt.show()

---
## 6. Deployment Notes

In [ ]:
model_path = MODEL_DIR / 'reintegration_model.pkl'
joblib.dump(best_pipeline, model_path)
print(f'Model saved: {model_path}')
print(f'Model: {best_model_name} | Test ROC-AUC: {test_aucs[best_model_name]:.4f}')

### API Usage

```bash
curl -X POST http://localhost:8001/predict/reintegration \
  -H 'Content-Type: application/json' \
  -d '{
    "avg_health_score_trend": 0.8,
    "avg_education_progress": 72.0,
    "incident_frequency": 0.1,
    "progress_noted_rate": 0.75,
    "counseling_session_count": 24,
    "days_in_program": 480,
    "initial_risk_level": "High",
    "sub_cat_trafficked": false,
    "sub_cat_physical_abuse": true,
    "sub_cat_sexual_abuse": false
  }'
```

**Response:**
```json
{
  "readiness_score": 0.73,
  "recommendation": "Ready for reintegration planning"
}
```

### Ethical Considerations
- This score is a **support tool**, never the sole decision-maker. A human social worker must review all cases.
- Sub-category features (trafficked, abuse) are sensitive — ensure the API is accessible only to authorized case workers.
- Retrain every 6 months as resident population characteristics evolve.

### Monitoring
- Track readiness score distribution monthly — alert if mean shifts by >0.1
- Validate against actual reintegration outcomes 90 days post-prediction